# ByteCore-DualMemory — 4-Hour Pre-Training (T4)

**Phase 25.3 durable Colab pipeline, end to end on a free T4.**

- **Model:** 8-layer, 1024-dim byte Transformer (**366,316,833 params**) with
  **causal sliding-window attention** over a **16,384-byte** packed row
  (each byte reads itself + up to **1024 previous bytes**), the original **Gated Skill Adapter**
  (1024 ↔ 8192 GLU) **plus context-to-memory cross-attention in every layer**
  (Q=1024, K=1024, V=8192), a confidence-headed adaptive refinement loop over
  the isolated Latent Thought Matrix, and one **unified dynamic memory bank**
  with learnable per-slot decay and guarded Forever slots.
- **Data mix (streaming, PT-BR first):** **50% Portuguese (Brazil)** · 25% code
  · 15% multimodal · 10% English, quota-exact per 100 draws. **100% streaming**:
  verified `IterableDataset` sources, RAM-only shuffle, zero-disk preflight,
  and actual **16,384-byte packed chunks**. The local window does not shorten
  the streamed row; the memory hub handles longer-range retrieval.
- **VRAM:** bf16 on Ampere+; **fp16 + GradScaler on T4**. Strict gradient
  checkpointing on all 8 blocks plus attention/FFN/adapter sub-boundaries,
  `batch_size=1`, `grad_accum=32`, a representative measured probe, and a hard
  **13.5 GB** ceiling. The analytic T4 estimate is **<10 GB**; the CUDA probe
  is the final authority. T4 uses memory-efficient SDPA (Flash/cuDNN SDPA
  require sm_80+). If the probe cannot fit, the run stops.
- **Checkpoints:** atomic local save + verified HF upload every **10 minutes**
  (or 100 optimizer steps), to `gabrielbosse9/airesearch2`. Logs/diagnostics
  sync every **5 minutes** and on normal exit/cooperative interruption. Re-running
  restores model, optimizer, scaler, memory bank, step and RNG state. Live HF
  stream cursors are reopened, not rewound to an exact document position.

> Runtime: **Runtime → Change runtime type → T4 GPU**. Add `HF_TOKEN` as a
> Colab secret with write access to `gabrielbosse9/airesearch2`, and enable
> notebook access for that secret. Never paste the token into code or chat.
> Wall clock ≈ 4 h: setup + 220 min training + eval/upload. A hard Colab
> disconnect can only be recovered from the last successful periodic upload.
> **Honesty:** the 16k row is not a 16k dense attention window. Self-attention
> is local; committed memory cross-attention is the long-range hand-off.


In [ ]:
# 1) GPU check — fail fast if the runtime is not a GPU.
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} (sm_{major}{minor}), {total_gb:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    print("T4 detected -> trainer resolves to float16 + GradScaler (required fallback)")

In [ ]:
# 2) Dependencies. torch/torchvision/torchaudio ship with Colab; add the rest.
import subprocess
import sys

packages = ["datasets>=3,<6", "huggingface_hub>=0.25", "pyyaml", "ddgs",
            "nbformat>=5", "tqdm", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("dependencies ready")

In [ ]:
# 3) Repository. Active delivery branch for the durable Colab fixes.
import os
import subprocess

REPO_URL = "https://github.com/gabrielbosse1/airesearch.git"
REPO_DIR = "/content/airesearch"
BRANCH = "arena/b49dc41e-airesearch"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(REPO_DIR)
print("working directory:", os.getcwd(), "| branch:", BRANCH)


In [ ]:
# 4) Hugging Face auth: use Colab Secrets, never a token literal in the notebook.
import getpass
import os
from pathlib import Path

from training.train import (DEFAULT_HF_REPO, hf_upload, redact_secrets,
                            resolve_hf_token, sync_run_folder, utc_stamp,
                            verify_hf_access, write_json)

HF_REPO = "gabrielbosse9/airesearch2"
HF_TOKEN_SECRET_NAME = "HF_TOKEN"       # Colab Secrets key with WRITE access
RUN_ID = os.environ.get("BYTECORE_RUN_ID") or f"colab-{utc_stamp()}"
RUN_DIR = Path("runs/colab_latest").resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_REPO"] = HF_REPO
os.environ["BYTECORE_RUN_ID"] = RUN_ID

token = resolve_hf_token()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get(HF_TOKEN_SECRET_NAME)
    except Exception:
        token = None
if not token:
    token = getpass.getpass("Hugging Face WRITE token (hidden; not saved to the notebook): ")
if not token:
    raise RuntimeError("HF_TOKEN is required for durable checkpoints/logs")
os.environ["HF_TOKEN"] = token          # inherited by the trainer; never printed

access = verify_hf_access(HF_REPO, token=token, run_dir=RUN_DIR)
print("HF write access verified:", access["repo_id"], "| user:", access["user"])
print("run id:", RUN_ID, "| local artifacts:", RUN_DIR)

def publish_preflight(name, payload, *, strict=True):
    path = RUN_DIR / f"preflight_{name}.json"
    write_json(path, {"run_id": RUN_ID, "name": name, "payload": payload})
    return hf_upload(path, HF_REPO, f"runs/{RUN_ID}/notebook/{name}.json",
                     token=resolve_hf_token(), strict=strict,
                     commit_message=f"Notebook preflight: {name}")


In [ ]:
# 5) Configuration + exact parameter count (meta device: zero memory cost).
import torch
import yaml

from models.foundation import CausalByteTransformer, FoundationConfig

with open("configs/t4_scaled.yaml", "r", encoding="utf8") as handle:
    config = yaml.safe_load(handle)
cfg = FoundationConfig.from_mapping(config["model"])
assert cfg.max_seq_len == config["model"]["max_seq_len"] == 16384
assert config["training"]["seq_len"] == 16384
assert config["training"]["micro_batch_size"] == 1
assert config["training"]["grad_accum"] == 32
assert cfg.gradient_checkpointing and cfg.attention_type == "sliding_window"
assert cfg.window_size == 1024 and cfg.sliding_chunk_size == 512
assert cfg.memory_cross_attention and config["vram"]["target_max_gb"] == 13.5
print("layers:", cfg.n_layers, "| d_model:", cfg.d_model, "| context:", cfg.max_seq_len)
print("sliding window:", cfg.window_size, "| chunk:", cfg.sliding_chunk_size)
print("frozen encoder dtype:", config["vram"]["frozen_encoder_dtype"], "| video backbone:", config["encoders"]["video"])
print("memory_dim:", cfg.memory_dim, "| cross-attention:", cfg.memory_cross_attention,
      "| refinement:", cfg.refinement_steps, "| confidence:", cfg.confidence_threshold)
with torch.device("meta"):
    probe = CausalByteTransformer(cfg)
print(f"parameters: {probe.parameters_count:,}")
assert probe.parameters_count == 366_316_833
print("data weights:", config["data"]["weights"])
assert config["checkpoints"]["hf_repo"] == "gabrielbosse9/airesearch2"
assert config["training"]["checkpoint_minutes"] == 10
assert config["checkpoints"]["upload_minutes"] == 5
print("checkpoint cadence: 10 min / 100 steps | diagnostics sync: 5 min")

# Chunking remains 16,384 bytes even though self-attention is local.
from data.sources import BytePacker
packer = BytePacker(int(config["training"]["seq_len"]))
rows = list(packer.feed(b"x" * (16384 + 1)))
assert rows and rows[0][0].shape[0] == 16384
print("streaming chunk length:", int(rows[0][0].shape[0]))


## Pre-flight: prove the PT-BR-first data mix

Portuguese (Brazil) is the primary language of the model, so the stream is
**50% PT-BR documents / 25% code / 15% multimodal / 10% English**.

`data/sources.MixtureSampler` allocates each **window of 100 draws** with the
largest-remainder method and shuffles it with the seeded RNG: 100 draws are
*exactly* 50/25/15/10, not merely on average. The cell below pulls **100 real
batches** through the streaming mixture and counts where every document came
from, so a broken source chain (e.g. COYO dead links) is visible **before** the
4-hour GPU budget is spent.

Every stream is shuffled with the iterable-safe
`.shuffle(seed=..., buffer_size=10000)` — a **RAM-only reservoir** (media
fallbacks capped at 1,000 decoded rows), never the map-style
`.shuffle(seed=...)` that fails on an `IterableDataset`. The
zero-disk cell after the audit then proves nothing is cached to disk.

Two numbers are printed, and they are different quantities: the **document
share** (the contract, which must match the weights) and the **byte share**
(informational — a Python file is far longer than an image caption, so bytes
will never match the document quota).

In [ ]:
# 6) PT-BR-first mix audit -- 100 real batches, before any training starts.
from data.sources import audit_mixture, build_mixture, format_mixture_report

train_cfg, data_cfg = config["training"], config["data"]
mix = build_mixture(data_cfg, seed=int(train_cfg["seed"]))
report = audit_mixture(
    mix,
    batches=int(train_cfg.get("mix_audit_batches", 100)),
    batch_size=int(train_cfg["micro_batch_size"]),
    tolerance=float(train_cfg.get("mix_audit_tolerance", 0.02)),
    max_seconds=180,                      # never hang the notebook on a slow chain
)
print(format_mixture_report(report))
print("effective weights:", mix.weights)
if report["failures"]:
    print("dropped source chains:", report["failures"])
publish_preflight("mix_audit", report)
assert report["quotas_ok"], "mix deviates from 50/25/15/10 -- fix the source chains above"
assert report["sample_share"]["ptbr"] >= 0.48, "PT-BR must dominate the stream"
print(f"OK: {report['samples']} documents, PT-BR share "
      f"{report['sample_share']['ptbr'] * 100:.1f}% (target 50.0%)")

## Zero-Disk verification (Phase 24.2)

`the-stack`, GigaVerbo and fineweb are **massive** — downloading them would
fill the Colab disk and crash the instance. The pipeline is therefore
**100% streaming**: `data/sources.open_stream` forces `streaming=True`,
verifies every source is an `IterableDataset` (a disk-backed `Dataset`
raises `TypeError` immediately), and shuffles with a RAM-only
`buffer_size=10000` reservoir (embedded-image/audio fallbacks are capped at
1,000 rows — decoded media rows are 100–1000× larger than text). The training loop is strictly **step-based**
(`while step < max_steps`) — `len(dataset)` and epochs never appear.

The cell below proves it on the live streams **before any training**:
it prints each group's dataset type, then measures disk usage before and
after the first **10 streamed batches** and asserts the growth is negligible
(≤ 32 MB — streaming metadata is KB-scale; a wholesale download is GB-scale).

In [ ]:
# 6b) ZERO-DISK verification (Phase 24.2): prove 100% streaming before training.
#     The Colab disk must NEVER fill with dataset files: every source must be
#     an IterableDataset, and 10 streamed batches must not grow disk usage.
import shutil

from datasets import IterableDataset

from data.sources import (DEFAULT_SOURCES, SHUFFLE_BUFFER_SIZE, MixtureSampler,
                          effective_shuffle_buffer, open_stream)

seed = int(train_cfg["seed"])
buffer_size = int(data_cfg.get("shuffle_buffer_size", SHUFFLE_BUFFER_SIZE))

# 1) Open each group's chain and print the stream TYPE (must be IterableDataset).
for group, specs in DEFAULT_SOURCES.items():
    if group not in mix.weights:
        print(f"{group:>10}: dropped by the audit above -- skipped")
        continue
    for spec in specs:
        try:
            stream = open_stream(spec, seed=seed, shuffle_buffer=buffer_size)
        except Exception as exc:
            print(f"{group:>10}: {spec.id} unavailable ({type(exc).__name__}) -- fallback")
            continue
        effective = effective_shuffle_buffer(spec, buffer_size)
        print(f"{group:>10}: {spec.id} -> {type(stream).__name__} "
              f"(shuffle buffer_size={effective:,})")
        assert isinstance(stream, IterableDataset), (
            f"{spec.id} returned {type(stream).__name__}: streaming=True violated")
        assert not hasattr(stream, "__len__"), "streaming datasets must not be sized"
        break
    else:
        raise AssertionError(f"group {group!r}: no streaming source could be opened")

assert isinstance(mix, MixtureSampler) and not hasattr(mix, "__len__")

# 2) Disk usage before/after the FIRST 10 streamed batches (no training yet).
def disk_used_mb() -> float:
    usage = shutil.disk_usage("/")   # Colab: /content and the HF cache live here
    return (usage.total - usage.free) / 1e6

before_mb = disk_used_mb()
docs = 0
for _ in range(10):
    docs += len(mix.next_batch(int(train_cfg["micro_batch_size"])))
after_mb = disk_used_mb()
growth_mb = after_mb - before_mb
print(f"disk used: {before_mb:,.1f} MB -> {after_mb:,.1f} MB "
      f"(growth {growth_mb:+.2f} MB over 10 batches / {docs} documents)")

# 3) Streaming metadata/lock files are KB-scale; a wholesale download is GB-scale.
MAX_GROWTH_MB = 32.0
publish_preflight("zero_disk", {"before_mb": before_mb, "after_mb": after_mb,
                                "growth_mb": growth_mb, "documents": docs,
                                "max_growth_mb": MAX_GROWTH_MB})
assert growth_mb <= MAX_GROWTH_MB, (
    f"disk grew {growth_mb:.1f} MB while streaming 10 batches -- data is being "
    "cached to disk; STOP and fix the pipeline before the 4-hour run")
print("OK: zero-disk streaming verified (IterableDataset types + negligible growth)")

## Training

One unified script: `python -m training.train`. It streams the PT-BR-first
50/25/15/10 mix in **16,384-byte chunks**, builds bidirectional multimodal
batches, and optimises causal-LM CE + confidence BCE + the unified memory
**decay/eviction policy loss**. Persistent writes commit only after the current
batch is assembled, so the teleportation hub cannot retrieve same-batch futures.

- Checkpoints: `runs/colab_latest/checkpoints/checkpoint_latest.pt`, every 10 min
  + every 100 steps, uploaded to `gabrielbosse9/airesearch2/checkpoints/checkpoint_latest.pt`.
  A normal exit also writes `checkpoints/checkpoint_final.pt`; upload failures
  stop the run by default instead of silently losing recoverability.
- The representative CUDA probe includes the full bank, local projector slots,
  confidence, policy gradients, all-block/sub-block checkpointing, and AdamW
  moments. It verifies the memory-efficient SDPA backend before measuring.
  Above **13.5 GB** it halves batch to 1 and reprobes; the 1024-byte window is
  an architecture contract and is not silently shrunk. If still over budget,
  it **fails closed** rather than inventing a fit.
- Logs, events, resolved config, source snapshot, VRAM probe, evaluation samples,
  and a checkpoint-free `debug_bundle.zip` sync every 5 min and on exit/crash.
- If Colab disconnects, re-run the notebook. Model/optimizer/scaler/memory/RNG
  resume from the latest successful checkpoint; live stream cursors are reopened,
  and incomplete gradient accumulation is discarded. A hard VM kill cannot run a
  final save, so the 10-minute cadence is the recovery bound.


In [ ]:
# 6c) Analytic T4 preflight -- explicitly a SIMULATION, not a CUDA measurement.
from training.train import simulate_t4_vram

sim = simulate_t4_vram(config)
print(f"SIMULATED T4 peak: {sim['peak_gb']:.3f} GB / {sim['target_max_gb']:.1f} GB")
print(sim["device"])
for name, gb in sim["components_gb"].items():
    print(f"  {name:38s} {gb:7.3f} GB")
assert sim["measured"] is False and sim["simulated"] is True
assert sim["fits"], "analytic estimate exceeds 13.5 GB -- do not launch; inspect components"
assert sim["meets_design_target"] and sim["peak_gb"] < 10.0, "Phase 25.1 design target is <10 GB"
print("backend:", sim["sdpa_backend_required"])
publish_preflight("t4_simulation", sim)
print("The real Trainer.probe_vram will measure CUDA peak before step 0 and may fail closed.")


In [ ]:
# 7) The 4-hour run: 220-minute budget, 10-minute checkpoints, 5-minute log sync.
import subprocess
import sys
import time

cmd = [sys.executable, "-u", "-m", "training.train",
       "--config", "configs/t4_scaled.yaml",
       "--hf-repo", HF_REPO,
       "--run-dir", str(RUN_DIR),
       "--run-id", RUN_ID,
       "--max-minutes", "220",
       "--checkpoint-minutes", "10",
       "--upload-minutes", "5"]
print("launching:", " ".join(cmd), flush=True)
start = time.time()
exit_code = None
launch_log = RUN_DIR / "notebook_launch.log"
with open(launch_log, "a", encoding="utf8") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=os.environ.copy())
    try:
        for line in proc.stdout:
            clean = redact_secrets(line)
            print(clean, end="", flush=True)
            log.write(clean)
            log.flush()
        exit_code = proc.wait()
    except BaseException:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
        raise
    finally:
        record = {"exit_code": exit_code, "elapsed_minutes": (time.time() - start) / 60,
                  "run_id": RUN_ID, "hf_repo": HF_REPO}
        write_json(RUN_DIR / "launch_exit.json", record)
        log.write(redact_secrets(str(record)) + "\n")
        log.flush()
        # Best-effort outer capture even if the child died before its own uploader.
        try:
            sync_run_folder(RUN_DIR, HF_REPO, f"runs/{RUN_ID}/notebook",
                            token=resolve_hf_token(), strict=False,
                            commit_message="Notebook launch console/exit diagnostics")
        except Exception as exc:
            print("outer diagnostics sync warning:", redact_secrets(str(exc)))
print(f"training exit code {exit_code} after {(time.time() - start) / 60:.1f} min")
assert exit_code == 0, "training failed -- diagnostics were retained locally and sync attempted"


In [ ]:
# 8) Post-training sanity from saved artifacts (no second multi-GB model load).
import json

summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf8"))
evaluation = json.loads((RUN_DIR / "evaluation.json").read_text(encoding="utf8"))
print("checkpoint step:", summary["steps"], "| stop:", summary["stop_reason"])
print("run:", summary["run_id"], "| remote:", summary["remote_run_path"])
for name, sample in evaluation.get("prompts", {}).items():
    print(f"\n{name}: {sample['prompt']!r} -> {sample['continuation']!r}")
    print("adaptive:", sample.get("adaptive"))
if "tool_demo" in evaluation:
    print("tool demo:", evaluation["tool_demo"])
print("debug bundle:", summary["debug_bundle"])


In [ ]:
# 9) Verify the final remote sync and expose the small debug bundle for download.
from training.train import make_debug_bundle, sync_run_folder

bundle = make_debug_bundle(RUN_DIR)
result = sync_run_folder(RUN_DIR, HF_REPO, f"runs/{RUN_ID}/notebook",
                         token=resolve_hf_token(), strict=True,
                         commit_message="Final notebook diagnostics")
assert result["uploaded"], result
checkpoint = summary["final_checkpoint"]
assert checkpoint.get("hf", {}).get("uploaded"), checkpoint
assert checkpoint.get("hf_final", {}).get("uploaded"), checkpoint
print("verified remote checkpoint:", checkpoint["hf"]["path_in_repo"])
print("verified remote final:", checkpoint["hf_final"]["path_in_repo"])
print("verified diagnostics:", f"{HF_REPO}/tree/main/{result['path_in_repo']}")
print("latest pointer:", f"{HF_REPO}/blob/main/runs/latest.json")
print("local debug bundle:", bundle)
# Optional browser download (small logs only, NOT the checkpoint):
# from google.colab import files
# files.download(str(bundle))


## Honest expectations

- 220 minutes on a T4 is a **small token budget** for a 366M-parameter model.
  Expect falling loss and byte-level structure, not a finished assistant.
- **16,384 bytes is the packed row, not the self-attention window.** Each byte
  attends locally to itself + up to 1024 previous bytes. Chunk-local SDPA masks
  avoid any `(L,L)` tensor. Information beyond that span must arrive through
  the committed memory cross-attention hub; there are no random/global edges.
- **VRAM numbers have two provenances:** cell 6c is an analytic *simulation*;
  the trainer log is a real CUDA `max_memory_allocated` measurement. The
  13.5 GB ceiling is enforced by the measured probe, with batch autotune and
  a hard stop if the sliding-window path still cannot fit.
- **Memory is unified and dynamic:** short slots score
  `importance − decay·age`; Forever slots have exactly zero effective decay,
  importance above the short range, and a hard eviction guard. The policy loss
  rewards stale eviction, discourages premature decay, and penalizes Forever
  overwrite attempts. Reads reinforce only explicitly used slots.
- **The mix is a document quota, not a token quota.** Multimodal documents
  expand into caption + codec rows; masking is source-agnostic (`PAD=258`).
- **Zero-disk streaming is verified, not assumed:** cell 6b asserts
  `IterableDataset` types and ≤32 MB growth over 10 streamed batches.
- **The Latent Thought Matrix is internal:** it never passes through the byte
  head; only the frontier reads it back. Cell 8 prints activation norm/gates.
- The T4 runs **fp16 + GradScaler**, not bf16 — resolved and logged at runtime.
- `[EDIT]` tokens are ordinary vocabulary; only `[TOOL_SEARCH]` ids execute a
  resource-limited subprocess.
- **Persistence is verified, not assumed:** checkpoint uploads fail closed by
  default; logs/config/source/provenance/evaluation sync separately. A hard
  Colab VM kill can lose only work since the last successful upload. Resume is
  exact for saved training state, not for the live HF stream cursor.
